In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW  # 修正：从 torch.optim 导入 AdamW

# 1. 加载模型和分词器
model_name = "facebook/opt-125m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. 加载数据集
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# 3. 分词并创建 DataLoader
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets.set_format("torch")
dataloader = DataLoader(tokenized_datasets, batch_size=10, shuffle=True)

# 4. 设置优化器
optimizer = AdamW(model.parameters(), lr=5e-5)  # 修正：使用 torch.optim.AdamW

# 5. 训练循环
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# device = torch.device("cpu")
model.to(device)
model.train()


from experiments.trainer.plugins import SnapshotPlugin, ProfilerPlugin
from perf_estimator.config import Config
snap_conf = Config(save2tmp=False)
snapshot = SnapshotPlugin(config=snap_conf)
profiler = ProfilerPlugin(config=snap_conf)

epochs = 1
snapshot.start()
profiler.start()
for epoch in range(epochs):
    for index, batch in enumerate(dataloader):
        snapshot.step()
        profiler.step()
        with torch.set_grad_enabled(True):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch, labels=batch["input_ids"])
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            print(f"Epoch {epoch}, Loss: {loss.item()}")
            if index == 3:
                break

snapshot.stop()
profiler.stop()

print("训练完成！")